# MLP para classificação de diabetes

Notebook autocontido. Toda a implementação está neste arquivo, organizada em funções pequenas, explícitas e executáveis em ordem.

## 1. Configuração

Valores do experimento e validações independentes.

In [ ]:
"""Configuração explícita do experimento."""

import os
from pathlib import Path


class ExperimentConfig:
    """Valores necessários para executar um experimento."""

    def __init__(
        self,
        project_root: Path,
        data_path: Path,
        artifacts_directory: Path,
        seed: int,
        training_fraction: float,
        validation_fraction: float,
        test_fraction: float,
        batch_size: int,
        epochs: int,
        learning_rate: float,
        weight_decay: float,
        hidden_dimensions: list[int],
        dropout: float,
        class_count: int,
        early_stopping_patience: int,
        log_interval: int,
        requested_device: str | None,
        maximum_rows: int | None,
        use_class_weights: bool,
    ) -> None:
        self.project_root = project_root
        self.data_path = data_path
        self.artifacts_directory = artifacts_directory
        self.seed = seed
        self.training_fraction = training_fraction
        self.validation_fraction = validation_fraction
        self.test_fraction = test_fraction
        self.batch_size = batch_size
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.hidden_dimensions = hidden_dimensions
        self.dropout = dropout
        self.class_count = class_count
        self.early_stopping_patience = early_stopping_patience
        self.log_interval = log_interval
        self.requested_device = requested_device
        self.maximum_rows = maximum_rows
        self.use_class_weights = use_class_weights


def create_default_config(project_root: Path) -> ExperimentConfig:
    data_path = project_root / "dataset" / "diabetes_prediction_dataset.csv"
    artifacts_directory = project_root / "artifacts"
    hidden_dimensions = [128, 64, 32]

    config = ExperimentConfig(
        project_root=project_root,
        data_path=data_path,
        artifacts_directory=artifacts_directory,
        seed=42,
        training_fraction=0.70,
        validation_fraction=0.15,
        test_fraction=0.15,
        batch_size=512,
        epochs=30,
        learning_rate=1e-3,
        weight_decay=1e-4,
        hidden_dimensions=hidden_dimensions,
        dropout=0.20,
        class_count=2,
        early_stopping_patience=6,
        log_interval=1,
        requested_device=None,
        maximum_rows=None,
        use_class_weights=False,
    )
    return config


def read_optional_integer_environment_variable(name: str) -> int | None:
    value = os.getenv(name)
    if value is None:
        return None
    if value == "":
        return None
    return int(value)


def read_boolean_environment_variable(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default

    normalized_value = value.strip().lower()
    true_values = ["1", "true", "yes"]
    if normalized_value in true_values:
        return True
    return False


def apply_environment_overrides(config: ExperimentConfig) -> ExperimentConfig:
    artifacts_value = os.getenv("MLP_ARTIFACTS_DIR")
    if artifacts_value is not None:
        if artifacts_value != "":
            config.artifacts_directory = Path(artifacts_value)

    epochs_value = read_optional_integer_environment_variable("MLP_EPOCHS")
    if epochs_value is not None:
        config.epochs = epochs_value

    batch_size_value = read_optional_integer_environment_variable("MLP_BATCH_SIZE")
    if batch_size_value is not None:
        config.batch_size = batch_size_value

    maximum_rows_value = read_optional_integer_environment_variable("MLP_MAX_ROWS")
    config.maximum_rows = maximum_rows_value

    device_value = os.getenv("MLP_DEVICE")
    if device_value is not None:
        if device_value == "":
            config.requested_device = None
        else:
            config.requested_device = device_value

    class_weights_value = read_boolean_environment_variable(
        "MLP_CLASS_WEIGHTS",
        config.use_class_weights,
    )
    config.use_class_weights = class_weights_value
    return config


def validate_split_proportions(config: ExperimentConfig) -> None:
    total = config.training_fraction
    total = total + config.validation_fraction
    total = total + config.test_fraction

    if abs(total - 1.0) > 1e-9:
        raise ValueError("As frações de treino, validação e teste devem somar 1.0.")

    proportions = [
        config.training_fraction,
        config.validation_fraction,
        config.test_fraction,
    ]
    for proportion in proportions:
        if proportion <= 0:
            raise ValueError("Todas as frações devem ser positivas.")


def validate_training_values(config: ExperimentConfig) -> None:
    if config.batch_size <= 0:
        raise ValueError("O tamanho do lote deve ser positivo.")
    if config.epochs <= 0:
        raise ValueError("O número de épocas deve ser positivo.")
    if config.learning_rate <= 0:
        raise ValueError("A taxa de aprendizado deve ser positiva.")
    if config.weight_decay < 0:
        raise ValueError("O weight decay não pode ser negativo.")
    if config.early_stopping_patience <= 0:
        raise ValueError("A paciência do early stopping deve ser positiva.")
    if config.log_interval <= 0:
        raise ValueError("O intervalo de log deve ser positivo.")
    if config.maximum_rows is not None:
        if config.maximum_rows <= 0:
            raise ValueError("O limite de linhas deve ser positivo.")


def validate_model_values(config: ExperimentConfig) -> None:
    if config.class_count != 2:
        raise ValueError("Este projeto espera exatamente duas classes.")
    if config.dropout < 0:
        raise ValueError("Dropout não pode ser negativo.")
    if config.dropout >= 1:
        raise ValueError("Dropout deve ser menor que 1.")
    if len(config.hidden_dimensions) == 0:
        raise ValueError("A MLP precisa de ao menos uma camada oculta.")

    for hidden_dimension in config.hidden_dimensions:
        if hidden_dimension <= 0:
            raise ValueError("As dimensões ocultas devem ser positivas.")


def validate_config(config: ExperimentConfig) -> None:
    validate_split_proportions(config)
    validate_training_values(config)
    validate_model_values(config)


def get_checkpoint_path(config: ExperimentConfig) -> Path:
    return config.artifacts_directory / "best_model.pt"


def get_preprocessor_path(config: ExperimentConfig) -> Path:
    return config.artifacts_directory / "preprocessor.joblib"


def config_to_dictionary(config: ExperimentConfig) -> dict[str, object]:
    values: dict[str, object] = {}
    values["project_root"] = str(config.project_root)
    values["data_path"] = str(config.data_path)
    values["artifacts_directory"] = str(config.artifacts_directory)
    values["seed"] = config.seed
    values["training_fraction"] = config.training_fraction
    values["validation_fraction"] = config.validation_fraction
    values["test_fraction"] = config.test_fraction
    values["batch_size"] = config.batch_size
    values["epochs"] = config.epochs
    values["learning_rate"] = config.learning_rate
    values["weight_decay"] = config.weight_decay
    values["hidden_dimensions"] = list(config.hidden_dimensions)
    values["dropout"] = config.dropout
    values["class_count"] = config.class_count
    values["early_stopping_patience"] = config.early_stopping_patience
    values["log_interval"] = config.log_interval
    values["requested_device"] = config.requested_device
    values["maximum_rows"] = config.maximum_rows
    values["use_class_weights"] = config.use_class_weights
    return values



## 2. Ambiente e reprodutibilidade

Sementes, determinismo e seleção explícita de CPU ou CUDA.

In [ ]:
"""Seleção de dispositivo e reprodutibilidade."""

import platform
import random
import sys

import numpy as np
import torch


class RuntimeMetadata:
    def __init__(
        self,
        python_version: str,
        platform_name: str,
        torch_version: str,
        torch_cuda_version: str | None,
        device_name: str,
        cuda_available: bool,
        gpu_name: str | None,
        cuda_device_count: int | None,
    ) -> None:
        self.python_version = python_version
        self.platform_name = platform_name
        self.torch_version = torch_version
        self.torch_cuda_version = torch_cuda_version
        self.device_name = device_name
        self.cuda_available = cuda_available
        self.gpu_name = gpu_name
        self.cuda_device_count = cuda_device_count

    def to_dictionary(self) -> dict[str, object]:
        values: dict[str, object] = {}
        values["python"] = self.python_version
        values["platform"] = self.platform_name
        values["torch"] = self.torch_version
        values["torch_cuda_version"] = self.torch_cuda_version
        values["device"] = self.device_name
        values["cuda_available"] = self.cuda_available
        if self.gpu_name is not None:
            values["gpu_name"] = self.gpu_name
        if self.cuda_device_count is not None:
            values["cuda_device_count"] = self.cuda_device_count
        return values


def seed_python(seed: int) -> None:
    random.seed(seed)


def seed_numpy(seed: int) -> None:
    np.random.seed(seed)


def seed_torch(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_deterministic_torch() -> None:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def configure_reproducibility(seed: int) -> None:
    seed_python(seed)
    seed_numpy(seed)
    seed_torch(seed)
    configure_deterministic_torch()


def validate_requested_device(device_name: str) -> None:
    requested_device = torch.device(device_name)
    if requested_device.type == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA foi solicitada, mas não está disponível.")


def select_device(device_name: str | None) -> torch.device:
    if device_name is not None:
        validate_requested_device(device_name)
        return torch.device(device_name)

    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def collect_runtime_metadata(device: torch.device) -> RuntimeMetadata:
    gpu_name = None
    cuda_device_count = None
    if device.type == "cuda":
        gpu_name = torch.cuda.get_device_name(device)
        cuda_device_count = torch.cuda.device_count()

    metadata = RuntimeMetadata(
        python_version=sys.version,
        platform_name=platform.platform(),
        torch_version=torch.__version__,
        torch_cuda_version=torch.version.cuda,
        device_name=str(device),
        cuda_available=torch.cuda.is_available(),
        gpu_name=gpu_name,
        cuda_device_count=cuda_device_count,
    )
    return metadata


def print_device_summary(metadata: RuntimeMetadata) -> None:
    print("Dispositivo selecionado: " + metadata.device_name)
    if metadata.gpu_name is not None:
        cuda_version = str(metadata.torch_cuda_version)
        print("GPU: " + metadata.gpu_name + " | CUDA PyTorch: " + cuda_version)
        return
    print("CUDA indisponível ou não solicitado; usando CPU.")



## 3. Leitura e divisão dos dados

Cada regra de validação do CSV possui uma função própria.

In [ ]:
"""Leitura, validação e divisão do dataset."""

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split



CATEGORICAL_COLUMNS = ["gender", "smoking_history"]
NUMERIC_COLUMNS = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
]
TARGET_COLUMN = "diabetes"
FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
REQUIRED_COLUMNS = FEATURE_COLUMNS + [TARGET_COLUMN]


class FeaturesAndTarget:
    def __init__(self, features: pd.DataFrame, target: pd.Series) -> None:
        self.features = features
        self.target = target


class RemainingAndTestData:
    def __init__(
        self,
        remaining_features: pd.DataFrame,
        test_features: pd.DataFrame,
        remaining_target: pd.Series,
        test_target: pd.Series,
    ) -> None:
        self.remaining_features = remaining_features
        self.test_features = test_features
        self.remaining_target = remaining_target
        self.test_target = test_target


class DatasetSplits:
    def __init__(
        self,
        x_train: pd.DataFrame,
        x_validation: pd.DataFrame,
        x_test: pd.DataFrame,
        y_train: pd.Series,
        y_validation: pd.Series,
        y_test: pd.Series,
    ) -> None:
        self.x_train = x_train
        self.x_validation = x_validation
        self.x_test = x_test
        self.y_train = y_train
        self.y_validation = y_validation
        self.y_test = y_test


class TargetSummary:
    def __init__(self, row_count: int, positive_rate: float, positive_count: int) -> None:
        self.row_count = row_count
        self.positive_rate = positive_rate
        self.positive_count = positive_count

    def to_dictionary(self) -> dict[str, float | int]:
        values: dict[str, float | int] = {}
        values["rows"] = self.row_count
        values["positive_rate"] = self.positive_rate
        values["positive_count"] = self.positive_count
        return values


class DatasetSplitSummary:
    def __init__(
        self,
        training: TargetSummary,
        validation: TargetSummary,
        test: TargetSummary,
    ) -> None:
        self.training = training
        self.validation = validation
        self.test = test

    def to_dictionary(self) -> dict[str, dict[str, float | int]]:
        values: dict[str, dict[str, float | int]] = {}
        values["train"] = self.training.to_dictionary()
        values["validation"] = self.validation.to_dictionary()
        values["test"] = self.test.to_dictionary()
        return values


class DatasetDiagnostics:
    def __init__(
        self,
        row_count: int,
        data_types: dict[str, str],
        null_counts: dict[str, int],
        target_distribution: dict[str, int],
    ) -> None:
        self.row_count = row_count
        self.data_types = data_types
        self.null_counts = null_counts
        self.target_distribution = target_distribution

    def to_dictionary(self) -> dict[str, object]:
        values: dict[str, object] = {}
        values["rows"] = self.row_count
        values["dtypes"] = self.data_types
        values["nulls"] = self.null_counts
        values["target_distribution"] = self.target_distribution
        return values


def ensure_dataset_file_exists(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError("Dataset não encontrado: " + str(path))


def read_dataset_csv(path: Path, maximum_rows: int | None) -> pd.DataFrame:
    return pd.read_csv(path, nrows=maximum_rows)


def find_missing_columns(
    frame: pd.DataFrame,
    required_columns: list[str],
) -> list[str]:
    missing_columns: list[str] = []
    for required_column in required_columns:
        if required_column not in frame.columns:
            missing_columns.append(required_column)
    missing_columns.sort()
    return missing_columns


def ensure_required_columns_exist(frame: pd.DataFrame) -> None:
    missing_columns = find_missing_columns(frame, REQUIRED_COLUMNS)
    if len(missing_columns) == 0:
        return

    found_columns = list(frame.columns)
    message = "CSV incompatível; colunas ausentes: " + str(missing_columns)
    message = message + ". Encontradas: " + str(found_columns)
    raise ValueError(message)


def ensure_dataset_is_not_empty(frame: pd.DataFrame) -> None:
    if frame.empty:
        raise ValueError("O dataset está vazio.")


def find_null_counts(frame: pd.DataFrame) -> dict[str, int]:
    counts: dict[str, int] = {}
    null_counts = frame[REQUIRED_COLUMNS].isna().sum()
    for column in REQUIRED_COLUMNS:
        count = int(null_counts[column])
        if count > 0:
            counts[column] = count
    return counts


def ensure_dataset_has_no_nulls(frame: pd.DataFrame) -> None:
    null_counts = find_null_counts(frame)
    if len(null_counts) == 0:
        return

    message = "O dataset contém valores ausentes. "
    message = message + "Política configurada: rejeitar. Detalhes: "
    message = message + str(null_counts)
    raise ValueError(message)


def ensure_target_is_binary(frame: pd.DataFrame) -> None:
    target = frame[TARGET_COLUMN]
    unique_values = target.unique().tolist()
    invalid_values: list[object] = []
    for value in unique_values:
        if value != 0:
            if value != 1:
                invalid_values.append(value)
    if len(invalid_values) > 0:
        invalid_values.sort()
        raise ValueError(
            "O alvo deve conter somente 0 e 1; encontrados: " + str(invalid_values)
        )


def ensure_target_has_both_classes(frame: pd.DataFrame) -> None:
    target = frame[TARGET_COLUMN]
    if target.nunique() != 2:
        raise ValueError("A divisão estratificada requer as duas classes no alvo.")


def select_model_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[REQUIRED_COLUMNS].copy()


def load_validated_dataset(path: Path, maximum_rows: int | None) -> pd.DataFrame:
    ensure_dataset_file_exists(path)
    frame = read_dataset_csv(path, maximum_rows)
    ensure_required_columns_exist(frame)
    ensure_dataset_is_not_empty(frame)
    ensure_dataset_has_no_nulls(frame)
    ensure_target_is_binary(frame)
    ensure_target_has_both_classes(frame)
    return select_model_columns(frame)


def separate_features_and_target(frame: pd.DataFrame) -> FeaturesAndTarget:
    features = frame.drop(columns=TARGET_COLUMN)
    target = frame[TARGET_COLUMN].astype("int64")
    return FeaturesAndTarget(features, target)


def calculate_relative_validation_size(config: ExperimentConfig) -> float:
    remaining_fraction = config.training_fraction + config.validation_fraction
    return config.validation_fraction / remaining_fraction


def split_test_set(
    features_and_target: FeaturesAndTarget,
    config: ExperimentConfig,
) -> RemainingAndTestData:
    split_values = train_test_split(
        features_and_target.features,
        features_and_target.target,
        test_size=config.test_fraction,
        random_state=config.seed,
        stratify=features_and_target.target,
    )
    remaining_features = split_values[0]
    test_features = split_values[1]
    remaining_target = split_values[2]
    test_target = split_values[3]
    return RemainingAndTestData(
        remaining_features,
        test_features,
        remaining_target,
        test_target,
    )


def split_training_and_validation(
    remaining_and_test: RemainingAndTestData,
    config: ExperimentConfig,
) -> DatasetSplits:
    validation_size = calculate_relative_validation_size(config)
    split_values = train_test_split(
        remaining_and_test.remaining_features,
        remaining_and_test.remaining_target,
        test_size=validation_size,
        random_state=config.seed,
        stratify=remaining_and_test.remaining_target,
    )
    training_features = split_values[0]
    validation_features = split_values[1]
    training_target = split_values[2]
    validation_target = split_values[3]

    return DatasetSplits(
        x_train=training_features,
        x_validation=validation_features,
        x_test=remaining_and_test.test_features,
        y_train=training_target,
        y_validation=validation_target,
        y_test=remaining_and_test.test_target,
    )


def create_dataset_splits(
    frame: pd.DataFrame,
    config: ExperimentConfig,
) -> DatasetSplits:
    features_and_target = separate_features_and_target(frame)
    remaining_and_test = split_test_set(features_and_target, config)
    return split_training_and_validation(remaining_and_test, config)


def summarize_target(target: pd.Series) -> TargetSummary:
    row_count = int(len(target))
    positive_rate = float(target.mean())
    positive_count = int(target.sum())
    return TargetSummary(row_count, positive_rate, positive_count)


def summarize_dataset_splits(splits: DatasetSplits) -> DatasetSplitSummary:
    training_summary = summarize_target(splits.y_train)
    validation_summary = summarize_target(splits.y_validation)
    test_summary = summarize_target(splits.y_test)
    return DatasetSplitSummary(training_summary, validation_summary, test_summary)


def create_dataset_diagnostics(frame: pd.DataFrame) -> DatasetDiagnostics:
    data_types: dict[str, str] = {}
    null_counts: dict[str, int] = {}
    target_distribution: dict[str, int] = {}

    for column in frame.columns:
        data_types[column] = str(frame[column].dtype)
        null_counts[column] = int(frame[column].isna().sum())

    distribution = frame[TARGET_COLUMN].value_counts().sort_index()
    for label in distribution.index:
        target_distribution[str(label)] = int(distribution[label])

    return DatasetDiagnostics(
        row_count=int(len(frame)),
        data_types=data_types,
        null_counts=null_counts,
        target_distribution=target_distribution,
    )



## 4. Pré-processamento

O pré-processador é ajustado somente no treino; validação e teste apenas transformam.

In [ ]:
"""Pré-processamento e criação explícita dos DataLoaders."""

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset



class TransformedSplits:
    def __init__(
        self,
        training: np.ndarray,
        validation: np.ndarray,
        test: np.ndarray,
    ) -> None:
        self.training = training
        self.validation = validation
        self.test = test


class DataLoaders:
    def __init__(
        self,
        train: DataLoader,
        validation: DataLoader,
        test: DataLoader,
    ) -> None:
        self.train = train
        self.validation = validation
        self.test = test


def create_categorical_transformer() -> OneHotEncoder:
    return OneHotEncoder(handle_unknown="ignore", sparse_output=False)


def create_numeric_transformer() -> Pipeline:
    steps = [("scaler", StandardScaler())]
    return Pipeline(steps)


def create_preprocessor() -> ColumnTransformer:
    categorical_transformer = create_categorical_transformer()
    numeric_transformer = create_numeric_transformer()
    transformers = [
        ("categorical", categorical_transformer, CATEGORICAL_COLUMNS),
        ("numeric", numeric_transformer, NUMERIC_COLUMNS),
    ]
    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.0,
    )


def convert_matrix_to_float32(matrix: object) -> np.ndarray:
    return np.asarray(matrix, dtype=np.float32)


def fit_and_transform_training_data(
    preprocessor: ColumnTransformer,
    training_features: pd.DataFrame,
) -> np.ndarray:
    transformed = preprocessor.fit_transform(training_features)
    return convert_matrix_to_float32(transformed)


def transform_validation_data(
    preprocessor: ColumnTransformer,
    validation_features: pd.DataFrame,
) -> np.ndarray:
    transformed = preprocessor.transform(validation_features)
    return convert_matrix_to_float32(transformed)


def transform_test_data(
    preprocessor: ColumnTransformer,
    test_features: pd.DataFrame,
) -> np.ndarray:
    transformed = preprocessor.transform(test_features)
    return convert_matrix_to_float32(transformed)


def ensure_equal_feature_widths(transformed_splits: TransformedSplits) -> None:
    training_width = transformed_splits.training.shape[1]
    validation_width = transformed_splits.validation.shape[1]
    test_width = transformed_splits.test.shape[1]
    if training_width == validation_width:
        if training_width == test_width:
            return

    widths: dict[str, int] = {}
    widths["train"] = training_width
    widths["validation"] = validation_width
    widths["test"] = test_width
    raise RuntimeError("Dimensões transformadas incompatíveis: " + str(widths))


def transform_dataset_splits(
    splits: DatasetSplits,
    preprocessor: ColumnTransformer,
) -> TransformedSplits:
    training = fit_and_transform_training_data(preprocessor, splits.x_train)
    validation = transform_validation_data(preprocessor, splits.x_validation)
    test = transform_test_data(preprocessor, splits.x_test)
    transformed_splits = TransformedSplits(training, validation, test)
    ensure_equal_feature_widths(transformed_splits)
    return transformed_splits


def convert_labels_to_int64(target: pd.Series) -> np.ndarray:
    return target.to_numpy(dtype=np.int64, copy=True)


def create_tensor_dataset(
    features: np.ndarray,
    labels: np.ndarray,
) -> TensorDataset:
    feature_tensor = torch.from_numpy(features)
    label_tensor = torch.from_numpy(labels)
    return TensorDataset(feature_tensor, label_tensor)


def create_training_generator(seed: int) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def create_data_loader(
    dataset: TensorDataset,
    batch_size: int,
    shuffle: bool,
    pin_memory: bool,
    generator: torch.Generator | None,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=pin_memory,
        generator=generator,
        num_workers=0,
    )


def create_data_loaders(
    transformed_splits: TransformedSplits,
    splits: DatasetSplits,
    config: ExperimentConfig,
    device: torch.device,
) -> DataLoaders:
    training_labels = convert_labels_to_int64(splits.y_train)
    validation_labels = convert_labels_to_int64(splits.y_validation)
    test_labels = convert_labels_to_int64(splits.y_test)

    training_dataset = create_tensor_dataset(
        transformed_splits.training,
        training_labels,
    )
    validation_dataset = create_tensor_dataset(
        transformed_splits.validation,
        validation_labels,
    )
    test_dataset = create_tensor_dataset(transformed_splits.test, test_labels)

    pin_memory = device.type == "cuda"
    training_generator = create_training_generator(config.seed)
    training_loader = create_data_loader(
        dataset=training_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        pin_memory=pin_memory,
        generator=training_generator,
    )
    validation_loader = create_data_loader(
        dataset=validation_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        generator=None,
    )
    test_loader = create_data_loader(
        dataset=test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        generator=None,
    )
    return DataLoaders(training_loader, validation_loader, test_loader)


def get_transformed_feature_count(transformed_splits: TransformedSplits) -> int:
    return int(transformed_splits.training.shape[1])



## 5. Modelo

A MLP é montada em blocos Linear, BatchNorm, ReLU e Dropout.

In [ ]:
"""Definição explícita da MLP."""

import torch
from torch import nn



def validate_hidden_dimensions(hidden_dimensions: list[int]) -> None:
    if len(hidden_dimensions) == 0:
        raise ValueError("A MLP precisa de ao menos uma dimensão oculta.")
    for hidden_dimension in hidden_dimensions:
        if hidden_dimension <= 0:
            raise ValueError("As dimensões ocultas devem ser positivas.")


def create_hidden_block(
    input_size: int,
    output_size: int,
    dropout: float,
) -> list[nn.Module]:
    modules: list[nn.Module] = []
    modules.append(nn.Linear(input_size, output_size))
    modules.append(nn.BatchNorm1d(output_size))
    modules.append(nn.ReLU())
    modules.append(nn.Dropout(dropout))
    return modules


def create_network_layers(
    input_size: int,
    hidden_dimensions: list[int],
    class_count: int,
    dropout: float,
) -> list[nn.Module]:
    validate_hidden_dimensions(hidden_dimensions)
    layers: list[nn.Module] = []
    previous_size = input_size

    for hidden_dimension in hidden_dimensions:
        hidden_block = create_hidden_block(
            input_size=previous_size,
            output_size=hidden_dimension,
            dropout=dropout,
        )
        for module in hidden_block:
            layers.append(module)
        previous_size = hidden_dimension

    output_layer = nn.Linear(previous_size, class_count)
    layers.append(output_layer)
    return layers


class MLP(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_dimensions: list[int],
        class_count: int,
        dropout: float,
    ) -> None:
        super().__init__()
        layers = create_network_layers(
            input_size=input_size,
            hidden_dimensions=hidden_dimensions,
            class_count=class_count,
            dropout=dropout,
        )
        self.network = nn.Sequential(*layers)
        self.input_size = input_size
        self.hidden_dimensions = list(hidden_dimensions)
        self.class_count = class_count
        self.dropout = dropout

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)


def create_model(
    input_size: int,
    config: ExperimentConfig,
    device: torch.device,
) -> MLP:
    model = MLP(
        input_size=input_size,
        hidden_dimensions=config.hidden_dimensions,
        class_count=config.class_count,
        dropout=config.dropout,
    )
    model.to(device)
    return model


def validate_model_output(
    model: MLP,
    sample_features: torch.Tensor,
    class_count: int,
    device: torch.device,
) -> None:
    model.eval()
    features_on_device = sample_features.to(device)
    with torch.no_grad():
        logits = model(features_on_device)

    expected_shape = (sample_features.shape[0], class_count)
    if logits.shape != expected_shape:
        message = "Formato de saída inválido. Esperado: " + str(expected_shape)
        message = message + ". Recebido: " + str(tuple(logits.shape))
        raise RuntimeError(message)
    if not torch.isfinite(logits).all():
        raise RuntimeError("A saída do modelo contém valores não finitos.")



## 6. Treinamento

Lote, época, checkpoint e early stopping são responsabilidades separadas.

In [ ]:
"""Treino, validação, checkpoint e early stopping."""

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader



class DeviceBatch:
    def __init__(self, features: torch.Tensor, labels: torch.Tensor) -> None:
        self.features = features
        self.labels = labels


class BatchTrainingResult:
    def __init__(self, weighted_loss: float, example_count: int) -> None:
        self.weighted_loss = weighted_loss
        self.example_count = example_count


class BatchPrediction:
    def __init__(
        self,
        weighted_loss: float,
        example_count: int,
        actual_labels: list[int],
        predicted_labels: list[int],
    ) -> None:
        self.weighted_loss = weighted_loss
        self.example_count = example_count
        self.actual_labels = actual_labels
        self.predicted_labels = predicted_labels


class EpochResult:
    def __init__(
        self,
        average_loss: float,
        actual_labels: np.ndarray | None,
        predicted_labels: np.ndarray | None,
    ) -> None:
        self.average_loss = average_loss
        self.actual_labels = actual_labels
        self.predicted_labels = predicted_labels


class TrainingHistory:
    def __init__(self) -> None:
        self.training_losses: list[float] = []
        self.validation_losses: list[float] = []
        self.validation_accuracies: list[float] = []
        self.validation_f1_scores: list[float] = []

    def add_epoch(
        self,
        training_loss: float,
        validation_loss: float,
        validation_accuracy: float,
        validation_f1: float,
    ) -> None:
        self.training_losses.append(training_loss)
        self.validation_losses.append(validation_loss)
        self.validation_accuracies.append(validation_accuracy)
        self.validation_f1_scores.append(validation_f1)

    def to_dictionary(self) -> dict[str, list[float]]:
        values: dict[str, list[float]] = {}
        values["train_loss"] = list(self.training_losses)
        values["val_loss"] = list(self.validation_losses)
        values["val_acc"] = list(self.validation_accuracies)
        values["val_f1"] = list(self.validation_f1_scores)
        return values


def move_batch_to_device(
    batch: list[torch.Tensor] | tuple[torch.Tensor, torch.Tensor],
    device: torch.device,
) -> DeviceBatch:
    non_blocking = device.type == "cuda"
    features = batch[0].to(device, non_blocking=non_blocking)
    labels = batch[1].to(device, non_blocking=non_blocking)
    return DeviceBatch(features, labels)


def calculate_batch_size(labels: torch.Tensor) -> int:
    return int(labels.size(0))


def train_single_batch(
    model: nn.Module,
    batch: list[torch.Tensor] | tuple[torch.Tensor, torch.Tensor],
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> BatchTrainingResult:
    device_batch = move_batch_to_device(batch, device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(device_batch.features)
    loss = loss_function(logits, device_batch.labels)
    loss.backward()
    optimizer.step()

    example_count = calculate_batch_size(device_batch.labels)
    weighted_loss = float(loss.item()) * example_count
    return BatchTrainingResult(weighted_loss, example_count)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> EpochResult:
    model.train()
    total_weighted_loss = 0.0
    total_examples = 0

    for batch in loader:
        batch_result = train_single_batch(
            model=model,
            batch=batch,
            loss_function=loss_function,
            optimizer=optimizer,
            device=device,
        )
        total_weighted_loss = total_weighted_loss + batch_result.weighted_loss
        total_examples = total_examples + batch_result.example_count

    if total_examples == 0:
        raise RuntimeError("Não é possível treinar com um DataLoader vazio.")
    average_loss = total_weighted_loss / total_examples
    return EpochResult(average_loss, None, None)


def predict_single_batch(
    model: nn.Module,
    batch: list[torch.Tensor] | tuple[torch.Tensor, torch.Tensor],
    loss_function: nn.Module,
    device: torch.device,
) -> BatchPrediction:
    device_batch = move_batch_to_device(batch, device)
    logits = model(device_batch.features)
    loss = loss_function(logits, device_batch.labels)
    predictions = logits.argmax(dim=1)

    example_count = calculate_batch_size(device_batch.labels)
    weighted_loss = float(loss.item()) * example_count
    actual_labels = device_batch.labels.cpu().tolist()
    predicted_labels = predictions.cpu().tolist()
    return BatchPrediction(
        weighted_loss=weighted_loss,
        example_count=example_count,
        actual_labels=actual_labels,
        predicted_labels=predicted_labels,
    )


def evaluate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
) -> EpochResult:
    model.eval()
    total_weighted_loss = 0.0
    total_examples = 0
    actual_labels: list[int] = []
    predicted_labels: list[int] = []

    with torch.no_grad():
        for batch in loader:
            batch_prediction = predict_single_batch(
                model=model,
                batch=batch,
                loss_function=loss_function,
                device=device,
            )
            total_weighted_loss = total_weighted_loss + batch_prediction.weighted_loss
            total_examples = total_examples + batch_prediction.example_count
            for label in batch_prediction.actual_labels:
                actual_labels.append(label)
            for prediction in batch_prediction.predicted_labels:
                predicted_labels.append(prediction)

    if total_examples == 0:
        raise RuntimeError("Não é possível avaliar com um DataLoader vazio.")

    average_loss = total_weighted_loss / total_examples
    actual_array = np.asarray(actual_labels, dtype=np.int64)
    predicted_array = np.asarray(predicted_labels, dtype=np.int64)
    return EpochResult(average_loss, actual_array, predicted_array)


def create_class_weights(
    training_target: pd.Series,
    class_count: int,
) -> torch.Tensor:
    target_values = training_target.to_numpy(dtype=np.int64, copy=True)
    counts = np.bincount(target_values, minlength=class_count)
    total_examples = len(target_values)
    weights: list[float] = []

    for class_index in range(class_count):
        class_examples = int(counts[class_index])
        if class_examples == 0:
            raise ValueError("Não é possível calcular peso para uma classe ausente.")
        denominator = class_count * class_examples
        class_weight = total_examples / denominator
        weights.append(class_weight)

    return torch.tensor(weights, dtype=torch.float32)


def create_loss_function(
    class_weights: torch.Tensor | None,
    device: torch.device,
) -> nn.CrossEntropyLoss:
    weights_on_device = None
    if class_weights is not None:
        weights_on_device = class_weights.to(device)
    return nn.CrossEntropyLoss(weight=weights_on_device)


def create_optimizer(
    model: nn.Module,
    config: ExperimentConfig,
) -> torch.optim.AdamW:
    return torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )


def validation_loss_improved(current_loss: float, best_loss: float) -> bool:
    return current_loss < best_loss


def should_stop_early(epochs_without_improvement: int, patience: int) -> bool:
    return epochs_without_improvement >= patience


def create_checkpoint_data(
    model: MLP,
    epoch: int,
    best_validation_loss: float,
    history: TrainingHistory,
    config: ExperimentConfig,
) -> dict[str, object]:
    checkpoint: dict[str, object] = {}
    checkpoint["format_version"] = 1
    checkpoint["state_dict"] = model.state_dict()
    checkpoint["epoch"] = epoch
    checkpoint["best_val_loss"] = best_validation_loss
    checkpoint["history"] = history.to_dictionary()
    checkpoint["in_dim"] = model.input_size
    checkpoint["hidden_dims"] = list(model.hidden_dimensions)
    checkpoint["num_classes"] = model.class_count
    checkpoint["dropout"] = model.dropout
    checkpoint["config"] = config_to_dictionary(config)
    return checkpoint


def save_best_checkpoint(checkpoint: dict[str, object], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, path)


def restore_best_model(model: nn.Module, path: Path, device: torch.device) -> None:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    state_dictionary = checkpoint["state_dict"]
    model.load_state_dict(state_dictionary)


def print_epoch_summary(
    epoch: int,
    training_loss: float,
    validation_loss: float,
    validation_accuracy: float,
    validation_f1: float,
) -> None:
    message = "epoch=" + format(epoch, "03d")
    message = message + " train_loss=" + format(training_loss, ".5f")
    message = message + " val_loss=" + format(validation_loss, ".5f")
    message = message + " val_acc=" + format(validation_accuracy, ".4f")
    message = message + " val_f1=" + format(validation_f1, ".4f")
    print(message)


def print_early_stopping(epoch: int, patience: int) -> None:
    message = "Early stopping na época " + str(epoch)
    message = message + "; paciência=" + str(patience) + "."
    print(message)


def train_model(
    model: MLP,
    training_loader: DataLoader,
    validation_loader: DataLoader,
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    config: ExperimentConfig,
    device: torch.device,
    checkpoint_path: Path,
) -> TrainingHistory:
    history = TrainingHistory()
    best_validation_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, config.epochs + 1):
        training_result = train_one_epoch(
            model=model,
            loader=training_loader,
            loss_function=loss_function,
            optimizer=optimizer,
            device=device,
        )
        validation_result = evaluate_one_epoch(
            model=model,
            loader=validation_loader,
            loss_function=loss_function,
            device=device,
        )
        if validation_result.actual_labels is None:
            raise RuntimeError("A validação não retornou os rótulos reais.")
        if validation_result.predicted_labels is None:
            raise RuntimeError("A validação não retornou as classes previstas.")

        validation_accuracy_value = accuracy_score(
            validation_result.actual_labels,
            validation_result.predicted_labels,
        )
        validation_accuracy = float(validation_accuracy_value)
        validation_f1_value = f1_score(
            validation_result.actual_labels,
            validation_result.predicted_labels,
            pos_label=1,
            zero_division=0,
        )
        validation_f1 = float(validation_f1_value)
        history.add_epoch(
            training_loss=training_result.average_loss,
            validation_loss=validation_result.average_loss,
            validation_accuracy=validation_accuracy,
            validation_f1=validation_f1,
        )

        if epoch % config.log_interval == 0:
            print_epoch_summary(
                epoch=epoch,
                training_loss=training_result.average_loss,
                validation_loss=validation_result.average_loss,
                validation_accuracy=validation_accuracy,
                validation_f1=validation_f1,
            )

        improved = validation_loss_improved(
            validation_result.average_loss,
            best_validation_loss,
        )
        if improved:
            best_validation_loss = validation_result.average_loss
            epochs_without_improvement = 0
            checkpoint = create_checkpoint_data(
                model=model,
                epoch=epoch,
                best_validation_loss=best_validation_loss,
                history=history,
                config=config,
            )
            save_best_checkpoint(checkpoint, checkpoint_path)
        else:
            epochs_without_improvement = epochs_without_improvement + 1

        stop = should_stop_early(
            epochs_without_improvement,
            config.early_stopping_patience,
        )
        if stop:
            print_early_stopping(epoch, config.early_stopping_patience)
            break

    restore_best_model(model, checkpoint_path, device)
    return history


## 7. Avaliação

As métricas são calculadas a partir de um único percurso pelo DataLoader.

In [ ]:
"""Cálculo explícito das métricas de classificação."""

import numpy as np
import torch
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from torch import nn
from torch.utils.data import DataLoader



class ClassificationMetrics:
    def __init__(
        self,
        loss: float,
        accuracy: float,
        positive_precision: float,
        positive_recall: float,
        positive_f1: float,
        report_dictionary: dict[str, object],
        report_text: str,
        confusion: np.ndarray,
    ) -> None:
        self.loss = loss
        self.accuracy = accuracy
        self.positive_precision = positive_precision
        self.positive_recall = positive_recall
        self.positive_f1 = positive_f1
        self.report_dictionary = report_dictionary
        self.report_text = report_text
        self.confusion = confusion

    def to_dictionary(self) -> dict[str, object]:
        values: dict[str, object] = {}
        values["loss"] = self.loss
        values["accuracy"] = self.accuracy
        values["precision_positive"] = self.positive_precision
        values["recall_positive"] = self.positive_recall
        values["f1_positive"] = self.positive_f1
        values["classification_report"] = self.report_dictionary
        return values


def calculate_accuracy(actual: np.ndarray, predicted: np.ndarray) -> float:
    return float(accuracy_score(actual, predicted))


def calculate_positive_precision(actual: np.ndarray, predicted: np.ndarray) -> float:
    value = precision_score(actual, predicted, pos_label=1, zero_division=0)
    return float(value)


def calculate_positive_recall(actual: np.ndarray, predicted: np.ndarray) -> float:
    value = recall_score(actual, predicted, pos_label=1, zero_division=0)
    return float(value)


def calculate_positive_f1(actual: np.ndarray, predicted: np.ndarray) -> float:
    value = f1_score(actual, predicted, pos_label=1, zero_division=0)
    return float(value)


def create_classification_report_dictionary(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, object]:
    report = classification_report(
        actual,
        predicted,
        output_dict=True,
        zero_division=0,
    )
    return report


def create_classification_report_text(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> str:
    return classification_report(actual, predicted, zero_division=0)


def create_confusion_matrix(actual: np.ndarray, predicted: np.ndarray) -> np.ndarray:
    labels = [0, 1]
    return confusion_matrix(actual, predicted, labels=labels)


def calculate_classification_metrics(epoch_result: EpochResult) -> ClassificationMetrics:
    if epoch_result.actual_labels is None:
        raise ValueError("O resultado da época não contém rótulos reais.")
    if epoch_result.predicted_labels is None:
        raise ValueError("O resultado da época não contém classes previstas.")

    actual = epoch_result.actual_labels
    predicted = epoch_result.predicted_labels
    accuracy = calculate_accuracy(actual, predicted)
    precision = calculate_positive_precision(actual, predicted)
    recall = calculate_positive_recall(actual, predicted)
    f1 = calculate_positive_f1(actual, predicted)
    report_dictionary = create_classification_report_dictionary(actual, predicted)
    report_text = create_classification_report_text(actual, predicted)
    confusion = create_confusion_matrix(actual, predicted)

    return ClassificationMetrics(
        loss=epoch_result.average_loss,
        accuracy=accuracy,
        positive_precision=precision,
        positive_recall=recall,
        positive_f1=f1,
        report_dictionary=report_dictionary,
        report_text=report_text,
        confusion=confusion,
    )


def evaluate_test_set(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
) -> ClassificationMetrics:
    epoch_result = evaluate_one_epoch(model, loader, loss_function, device)
    return calculate_classification_metrics(epoch_result)



## 8. Artefatos

Cada arquivo ou gráfico possui uma operação de persistência nomeada.

In [ ]:
"""Persistência dos artefatos produzidos pelo experimento."""

import json
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer



def ensure_artifacts_directory_exists(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def save_json_file(path: Path, value: dict[str, object]) -> None:
    serialized_value = json.dumps(value, indent=2, ensure_ascii=False)
    path.write_text(serialized_value, encoding="utf-8")


def save_text_file(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")


def save_preprocessor(path: Path, preprocessor: ColumnTransformer) -> None:
    joblib.dump(preprocessor, path)


def load_preprocessor(path: Path) -> ColumnTransformer:
    loaded_preprocessor = joblib.load(path)
    return loaded_preprocessor


def save_test_metrics(
    metrics: ClassificationMetrics,
    artifacts_directory: Path,
) -> None:
    path = artifacts_directory / "test_metrics.json"
    save_json_file(path, metrics.to_dictionary())


def save_classification_report(
    metrics: ClassificationMetrics,
    artifacts_directory: Path,
) -> None:
    path = artifacts_directory / "classification_report.txt"
    save_text_file(path, metrics.report_text)


def save_confusion_matrix_figure(
    metrics: ClassificationMetrics,
    artifacts_directory: Path,
) -> None:
    figure, axis = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        metrics.confusion,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[0, 1],
        yticklabels=[0, 1],
        ax=axis,
    )
    axis.set_xlabel("Predito")
    axis.set_ylabel("Real")
    axis.set_title("Matriz de confusão — teste")
    figure.tight_layout()
    path = artifacts_directory / "confusion_matrix.png"
    figure.savefig(path, dpi=160)
    plt.close(figure)


def save_learning_curves_figure(
    history: TrainingHistory,
    artifacts_directory: Path,
) -> None:
    epoch_count = len(history.training_losses)
    epochs = range(1, epoch_count + 1)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    loss_axis = axes[0]
    metrics_axis = axes[1]

    loss_axis.plot(epochs, history.training_losses, label="Treino")
    loss_axis.plot(epochs, history.validation_losses, label="Validação")
    loss_axis.set_title("Perda")
    loss_axis.set_xlabel("Época")
    loss_axis.set_ylabel("Cross-entropy")

    metrics_axis.plot(epochs, history.validation_accuracies, label="Acurácia")
    metrics_axis.plot(epochs, history.validation_f1_scores, label="F1 positiva")
    metrics_axis.set_title("Métricas de validação")
    metrics_axis.set_xlabel("Época")
    metrics_axis.set_ylabel("Valor")

    for axis in axes:
        axis.grid(True, alpha=0.3)
        axis.legend()

    figure.tight_layout()
    path = artifacts_directory / "learning_curves.png"
    figure.savefig(path, dpi=160)
    plt.close(figure)


def save_history(history: TrainingHistory, artifacts_directory: Path) -> None:
    path = artifacts_directory / "history.json"
    values: dict[str, object] = {}
    history_dictionary = history.to_dictionary()
    for key in history_dictionary:
        values[key] = history_dictionary[key]
    save_json_file(path, values)


def save_metadata(metadata: dict[str, object], artifacts_directory: Path) -> None:
    path = artifacts_directory / "metadata.json"
    save_json_file(path, metadata)


def build_imbalance_report(
    split_summary: DatasetSplitSummary,
    metrics: ClassificationMetrics,
) -> str:
    lines: list[str] = []
    training_rate = format(split_summary.training.positive_rate, ".5f")
    test_rate = format(split_summary.test.positive_rate, ".5f")
    lines.append("Taxa positiva treino: " + training_rate)
    lines.append("Taxa positiva teste: " + test_rate)
    lines.append("Acurácia: " + format(metrics.accuracy, ".5f"))
    lines.append("Precision positiva: " + format(metrics.positive_precision, ".5f"))
    lines.append("Recall positivo: " + format(metrics.positive_recall, ".5f"))
    lines.append("F1 positivo: " + format(metrics.positive_f1, ".5f"))
    lines.append(
        "Conclusão: em classes desbalanceadas, acurácia isolada não é critério suficiente."
    )
    return "\n".join(lines) + "\n"


def save_imbalance_report(
    split_summary: DatasetSplitSummary,
    metrics: ClassificationMetrics,
    artifacts_directory: Path,
) -> None:
    report = build_imbalance_report(split_summary, metrics)
    path = artifacts_directory / "imbalance_report.txt"
    save_text_file(path, report)


def save_experiment_artifacts(
    preprocessor: ColumnTransformer,
    preprocessor_path: Path,
    metrics: ClassificationMetrics,
    history: TrainingHistory,
    split_summary: DatasetSplitSummary,
    artifacts_directory: Path,
) -> None:
    ensure_artifacts_directory_exists(artifacts_directory)
    save_preprocessor(preprocessor_path, preprocessor)
    save_test_metrics(metrics, artifacts_directory)
    save_classification_report(metrics, artifacts_directory)
    save_confusion_matrix_figure(metrics, artifacts_directory)
    save_learning_curves_figure(history, artifacts_directory)
    save_history(history, artifacts_directory)
    save_imbalance_report(split_summary, metrics, artifacts_directory)



## 9. Inferência

Modelo e pré-processador são carregados uma vez e reutilizados.

In [ ]:
"""Carregamento de artefatos e inferência sem dados de treino."""

from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer



class Predictor:
    def __init__(
        self,
        model: MLP,
        preprocessor: ColumnTransformer,
        device: torch.device,
    ) -> None:
        self.model = model
        self.preprocessor = preprocessor
        self.device = device


def normalize_checkpoint_state_dictionary(
    state_dictionary: dict[str, torch.Tensor],
) -> OrderedDict[str, torch.Tensor]:
    normalized: OrderedDict[str, torch.Tensor] = OrderedDict()
    for key in state_dictionary:
        normalized_key = key
        if key.startswith("net."):
            normalized_key = "network." + key[4:]
        normalized[normalized_key] = state_dictionary[key]
    return normalized


def ensure_inference_columns_exist(frame: pd.DataFrame) -> None:
    missing_columns = find_missing_columns(frame, FEATURE_COLUMNS)
    if len(missing_columns) > 0:
        raise ValueError("Entradas sem colunas exigidas: " + str(missing_columns))


def select_inference_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[FEATURE_COLUMNS]


def load_model_from_checkpoint(path: Path, device: torch.device) -> MLP:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    input_size = int(checkpoint["in_dim"])
    hidden_dimensions = list(checkpoint["hidden_dims"])
    class_count = int(checkpoint["num_classes"])
    dropout = float(checkpoint["dropout"])

    model = MLP(
        input_size=input_size,
        hidden_dimensions=hidden_dimensions,
        class_count=class_count,
        dropout=dropout,
    )
    model.to(device)
    state_dictionary = checkpoint["state_dict"]
    normalized_state_dictionary = normalize_checkpoint_state_dictionary(
        state_dictionary
    )
    model.load_state_dict(normalized_state_dictionary)
    model.eval()
    return model


def transform_inference_features(
    frame: pd.DataFrame,
    preprocessor: ColumnTransformer,
) -> np.ndarray:
    selected_frame = select_inference_columns(frame)
    transformed = preprocessor.transform(selected_frame)
    return np.asarray(transformed, dtype=np.float32)


def predict_classes(
    model: MLP,
    transformed_features: np.ndarray,
    device: torch.device,
) -> np.ndarray:
    feature_tensor = torch.from_numpy(transformed_features)
    feature_tensor = feature_tensor.to(device)
    with torch.no_grad():
        logits = model(feature_tensor)
        predictions = logits.argmax(dim=1)
    return predictions.cpu().numpy()


def load_predictor(
    checkpoint_path: Path,
    preprocessor_path: Path,
    device: torch.device,
) -> Predictor:
    model = load_model_from_checkpoint(checkpoint_path, device)
    preprocessor = load_preprocessor(preprocessor_path)
    return Predictor(model, preprocessor, device)


def predict(predictor: Predictor, frame: pd.DataFrame) -> np.ndarray:
    ensure_inference_columns_exist(frame)
    transformed_features = transform_inference_features(
        frame,
        predictor.preprocessor,
    )
    return predict_classes(
        predictor.model,
        transformed_features,
        predictor.device,
    )


## 10. Orquestração

O fluxo principal apenas coordena as funções definidas anteriormente.

In [ ]:
"""Orquestração legível do experimento completo."""

import json
from pathlib import Path

import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from torch import nn
from torch.utils.data import DataLoader



class PreparedExperiment:
    def __init__(
        self,
        config: ExperimentConfig,
        device: torch.device,
        runtime_metadata: RuntimeMetadata,
    ) -> None:
        self.config = config
        self.device = device
        self.runtime_metadata = runtime_metadata


class PreparedTrainingData:
    def __init__(
        self,
        frame: pd.DataFrame,
        splits: DatasetSplits,
        split_summary: DatasetSplitSummary,
        diagnostics: DatasetDiagnostics,
        preprocessor: ColumnTransformer,
        loaders: DataLoaders,
        input_size: int,
    ) -> None:
        self.frame = frame
        self.splits = splits
        self.split_summary = split_summary
        self.diagnostics = diagnostics
        self.preprocessor = preprocessor
        self.loaders = loaders
        self.input_size = input_size


class TrainingComponents:
    def __init__(
        self,
        model: MLP,
        loss_function: nn.Module,
        optimizer: torch.optim.Optimizer,
    ) -> None:
        self.model = model
        self.loss_function = loss_function
        self.optimizer = optimizer


class ExperimentResult:
    def __init__(
        self,
        config: ExperimentConfig,
        runtime_metadata: RuntimeMetadata,
        diagnostics: DatasetDiagnostics,
        split_summary: DatasetSplitSummary,
        history: TrainingHistory,
        test_metrics: ClassificationMetrics,
        inference_predictions: list[int],
        input_size: int,
    ) -> None:
        self.config = config
        self.runtime_metadata = runtime_metadata
        self.diagnostics = diagnostics
        self.split_summary = split_summary
        self.history = history
        self.test_metrics = test_metrics
        self.inference_predictions = inference_predictions
        self.input_size = input_size

    def to_dictionary(self) -> dict[str, object]:
        values = self.runtime_metadata.to_dictionary()
        values["config"] = config_to_dictionary(self.config)
        values["diagnostics"] = self.diagnostics.to_dictionary()
        values["splits"] = self.split_summary.to_dictionary()
        values["history"] = self.history.to_dictionary()
        values["test_metrics"] = self.test_metrics.to_dictionary()
        values["inference_smoke_predictions"] = list(self.inference_predictions)
        values["class_weighting_enabled"] = self.config.use_class_weights
        values["in_dim"] = self.input_size
        values["imbalance_conclusion"] = (
            "Acurácia não é suficiente; recall e F1 da classe positiva "
            "devem ser avaliados junto dela."
        )
        return values


def prepare_experiment(config: ExperimentConfig) -> PreparedExperiment:
    validate_config(config)
    configure_reproducibility(config.seed)
    device = select_device(config.requested_device)
    runtime_metadata = collect_runtime_metadata(device)
    print_device_summary(runtime_metadata)
    ensure_artifacts_directory_exists(config.artifacts_directory)
    return PreparedExperiment(config, device, runtime_metadata)


def prepare_training_data(
    config: ExperimentConfig,
    device: torch.device,
) -> PreparedTrainingData:
    frame = load_validated_dataset(config.data_path, config.maximum_rows)
    diagnostics = create_dataset_diagnostics(frame)
    splits = create_dataset_splits(frame, config)
    split_summary = summarize_dataset_splits(splits)
    preprocessor = create_preprocessor()
    transformed_splits = transform_dataset_splits(splits, preprocessor)
    input_size = get_transformed_feature_count(transformed_splits)
    loaders = create_data_loaders(
        transformed_splits=transformed_splits,
        splits=splits,
        config=config,
        device=device,
    )
    return PreparedTrainingData(
        frame=frame,
        splits=splits,
        split_summary=split_summary,
        diagnostics=diagnostics,
        preprocessor=preprocessor,
        loaders=loaders,
        input_size=input_size,
    )


def print_data_summary(training_data: PreparedTrainingData) -> None:
    summary = training_data.split_summary.to_dictionary()
    message = "Partições: " + str(summary)
    message = message + "; in_dim=" + str(training_data.input_size)
    print(message)


def get_sample_features(training_loader: DataLoader) -> torch.Tensor:
    iterator = iter(training_loader)
    sample_batch = next(iterator)
    return sample_batch[0]


def validate_prepared_model(
    model: MLP,
    training_data: PreparedTrainingData,
    config: ExperimentConfig,
    device: torch.device,
) -> None:
    sample_features = get_sample_features(training_data.loaders.train)
    validate_model_output(model, sample_features, config.class_count, device)
    if device.type == "cuda":
        if not training_data.loaders.train.pin_memory:
            raise RuntimeError("pin_memory deveria estar ativo em CUDA.")


def prepare_training_components(
    training_data: PreparedTrainingData,
    config: ExperimentConfig,
    device: torch.device,
) -> TrainingComponents:
    model = create_model(training_data.input_size, config, device)
    validate_prepared_model(model, training_data, config, device)

    class_weights = None
    if config.use_class_weights:
        class_weights = create_class_weights(
            training_data.splits.y_train,
            config.class_count,
        )

    loss_function = create_loss_function(class_weights, device)
    optimizer = create_optimizer(model, config)
    return TrainingComponents(model, loss_function, optimizer)


def run_training(
    components: TrainingComponents,
    training_data: PreparedTrainingData,
    prepared_experiment: PreparedExperiment,
) -> TrainingHistory:
    config = prepared_experiment.config
    checkpoint_path = get_checkpoint_path(config)
    return train_model(
        model=components.model,
        training_loader=training_data.loaders.train,
        validation_loader=training_data.loaders.validation,
        loss_function=components.loss_function,
        optimizer=components.optimizer,
        config=config,
        device=prepared_experiment.device,
        checkpoint_path=checkpoint_path,
    )


def run_test_evaluation(
    components: TrainingComponents,
    training_data: PreparedTrainingData,
    device: torch.device,
) -> ClassificationMetrics:
    return evaluate_test_set(
        model=components.model,
        loader=training_data.loaders.test,
        loss_function=components.loss_function,
        device=device,
    )


def persist_primary_artifacts(
    training_data: PreparedTrainingData,
    history: TrainingHistory,
    metrics: ClassificationMetrics,
    config: ExperimentConfig,
) -> None:
    preprocessor_path = get_preprocessor_path(config)
    save_experiment_artifacts(
        preprocessor=training_data.preprocessor,
        preprocessor_path=preprocessor_path,
        metrics=metrics,
        history=history,
        split_summary=training_data.split_summary,
        artifacts_directory=config.artifacts_directory,
    )


def verify_saved_inference(
    training_data: PreparedTrainingData,
    config: ExperimentConfig,
    device: torch.device,
) -> list[int]:
    checkpoint_path = get_checkpoint_path(config)
    preprocessor_path = get_preprocessor_path(config)
    predictor = load_predictor(checkpoint_path, preprocessor_path, device)
    sample = training_data.splits.x_test.head(3)
    predictions = predict(predictor, sample)
    return predictions.tolist()


def build_experiment_result(
    prepared_experiment: PreparedExperiment,
    training_data: PreparedTrainingData,
    history: TrainingHistory,
    metrics: ClassificationMetrics,
    inference_predictions: list[int],
) -> ExperimentResult:
    return ExperimentResult(
        config=prepared_experiment.config,
        runtime_metadata=prepared_experiment.runtime_metadata,
        diagnostics=training_data.diagnostics,
        split_summary=training_data.split_summary,
        history=history,
        test_metrics=metrics,
        inference_predictions=inference_predictions,
        input_size=training_data.input_size,
    )


def print_experiment_result(result: ExperimentResult) -> None:
    metrics_json = json.dumps(
        result.test_metrics.to_dictionary(),
        ensure_ascii=False,
    )
    print("Métricas de teste: " + metrics_json)
    print("Inferência recarregada (3 amostras): " + str(result.inference_predictions))


def run_experiment(config: ExperimentConfig) -> ExperimentResult:
    prepared_experiment = prepare_experiment(config)
    training_data = prepare_training_data(config, prepared_experiment.device)
    print_data_summary(training_data)
    components = prepare_training_components(
        training_data,
        config,
        prepared_experiment.device,
    )
    history = run_training(components, training_data, prepared_experiment)
    metrics = run_test_evaluation(
        components,
        training_data,
        prepared_experiment.device,
    )
    persist_primary_artifacts(training_data, history, metrics, config)
    predictions = verify_saved_inference(
        training_data,
        config,
        prepared_experiment.device,
    )
    result = build_experiment_result(
        prepared_experiment,
        training_data,
        history,
        metrics,
        predictions,
    )
    save_metadata(result.to_dictionary(), config.artifacts_directory)
    print_experiment_result(result)
    return result


def results_are_identical(
    first_result: ExperimentResult,
    second_result: ExperimentResult,
) -> bool:
    first_values = first_result.to_dictionary()
    second_values = second_result.to_dictionary()
    keys_to_compare = [
        "splits",
        "history",
        "test_metrics",
        "inference_smoke_predictions",
    ]
    for key in keys_to_compare:
        if first_values[key] != second_values[key]:
            return False
    return True


def verify_reproducibility(
    project_root: Path,
    maximum_rows: int,
    epochs: int,
) -> dict[str, object]:
    verification_directory = (
        project_root / "artifacts" / "refactor_reproducibility"
    )
    first_config = create_default_config(project_root)
    first_config.requested_device = "cpu"
    first_config.maximum_rows = maximum_rows
    first_config.epochs = epochs
    first_config.artifacts_directory = verification_directory / "run_1"

    second_config = create_default_config(project_root)
    second_config.requested_device = "cpu"
    second_config.maximum_rows = maximum_rows
    second_config.epochs = epochs
    second_config.artifacts_directory = verification_directory / "run_2"

    first_result = run_experiment(first_config)
    second_result = run_experiment(second_config)
    if not results_are_identical(first_result, second_result):
        raise AssertionError("Execuções CPU com a mesma semente divergiram.")

    summary: dict[str, object] = {}
    summary["device"] = "cpu"
    summary["maximum_rows"] = maximum_rows
    summary["epochs"] = epochs
    summary["splits"] = first_result.split_summary.to_dictionary()
    summary["test_metrics"] = first_result.test_metrics.to_dictionary()
    summary["result"] = "identical deterministic CPU runs"
    path = verification_directory / "reproducibility_check.json"
    path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


def verify_cuda(
    project_root: Path,
    maximum_rows: int,
    epochs: int,
) -> ExperimentResult:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA não está disponível neste ambiente.")
    config = create_default_config(project_root)
    config.requested_device = "cuda"
    config.maximum_rows = maximum_rows
    config.epochs = epochs
    config.artifacts_directory = project_root / "artifacts" / "refactor_cuda_validation"
    return run_experiment(config)


## 11. Configurar a execução

A execução abaixo é curta para facilitar a inspeção. Para o treino completo, use `maximum_rows = None` e `epochs = 30`.

In [ ]:
PROJECT_ROOT = Path.cwd()
config = create_default_config(PROJECT_ROOT)
config.maximum_rows = 3000
config.epochs = 3
config.artifacts_directory = PROJECT_ROOT / "artifacts" / "notebook_validation"
validate_config(config)
config_to_dictionary(config)


## 12. Executar o experimento

Esta chamada realiza preparação, treino, avaliação, persistência e teste de inferência.

In [ ]:
result = run_experiment(config)
result.to_dictionary()


## 13. Validações opcionais

As chamadas permanecem comentadas porque executam treinamentos adicionais.

In [ ]:
# reproducibility_result = verify_reproducibility(PROJECT_ROOT, maximum_rows=3000, epochs=3)
# cuda_result = verify_cuda(PROJECT_ROOT, maximum_rows=3000, epochs=3)
